# 03-2 · The four C1 selectors

| | |
|---|---|
| **What it does** | collapses each option's row of S into one number, four different ways |
| **Proves** | the four selectors are genuinely different functions — Table 1 of the paper |
| **Ported verbatim from** | Vincent (`wangvincentx-cmd`), `backend/services/fairness_service.py`, commit `3c32c76`, "Add per-member F0 taste vectors and C1 fairness aggregation selectors" (2026-08-02) |
| **Eval surface** | run all four on the same S and watch them disagree |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## The four selectors, read directly from `nbio.py` (Vincent's code, unchanged)

```python
def average_score(scores):       return sum(scores) / len(scores)
def least_misery_score(scores):  return min(scores)
def most_pleasure_score(scores): return max(scores)
def fairness_score(scores, lam): return average_score(scores) - lam * variance(scores)
```
`lambda=0` makes `fair` identical to `average`; raising it trades mean
satisfaction for a flatter spread. This is Vincent's own docstring, verbatim:

> average — highest average happiness, can leave individuals behind
> least_misery — protects the worst-off member, but drags the group to the safest option
> most_pleasure — favors whoever is most excited, ignores everyone else
> fair — average minus a fairness penalty for spread between members


In [3]:
# Step 1: rebuild S from 03-1 (same 4 members, same catalog)
personas = json.load(open(nbio.DATA / "personas.json"))
group_names = ["Vihaan", "Vincent", "Vasu", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"], "taste": p["taste_vector"]})

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        n = row.get("ingredient", "").strip().lower()
        if n: flavor_map[n] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip(); return
        for v in node.values(): yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node: yield from iter_dish_names(v)

def infer_taste(d):
    m = [flavor_map[i] for i in flavor_map if i in d.lower()]
    return [sum(x[k] for x in m)/len(m) for k in nbio.TASTE_DIMS] if m else [0.0]*6

catalog = []
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    names = [n for n in iter_dish_names(json.loads(f.read_text())) if n]
    vecs = [infer_taste(n) for n in names]
    nz = [v for v in vecs if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6
    catalog.append({"option_id": f.stem, "name": f.stem, "menu": [{"name": n} for n in names], "taste": avg})

F = nbio.build_feasible_set(catalog, members)
S = {o["option_id"]: {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"]) for m in members} for o in F}
labels = {o["option_id"]: o["name"] for o in F}
print(f"F size: {len(F)}")


F size: 5


## Score every option, all four ways


In [4]:
# Step 2: aggregate() dispatches to Vincent's four selectors (nbio.py, ported verbatim)
SELECTORS = ["average", "least_misery", "most_pleasure", "fair"]
rows = []
for oid, row in S.items():
    scores = list(row.values())
    entry = {"option": labels[oid][:26]}
    for sel in SELECTORS:
        entry[sel] = nbio.aggregate(scores, method=sel, lam=0.5)
    rows.append(entry)
nbio.table(rows, cols=["option", *SELECTORS], width=26)


option                      average  least_misery  most_pleasure  fair 
-----------------------------------------------------------------------
Arabian Corner Restaurant   0.658    0.501         0.787          0.652
Casa.cafe                   0.761    0.624         0.849          0.757
Delicia Multi Cuisine Rest  0.759    0.622         0.834          0.755
House of Chefs              0.705    0.540         0.801          0.699
Ministry of South Bar and   0.689    0.529         0.801          0.683


## Pick the winner under each selector


In [5]:
# Step 3: nbio.rank_candidates (Vincent's rank_candidates/select_best, ported)
candidate_vectors = {o["option_id"]: o["taste"] for o in F}
member_vectors = [m["taste"] for m in members]

for sel in SELECTORS:
    ranked = nbio.rank_candidates(candidate_vectors, member_vectors, method=sel, lam=0.5)
    winner_id, score = ranked[0]
    print(f"   {sel:<14} -> {labels[winner_id]:<34} score={score:.4f}")


   average        -> Casa.cafe                          score=0.7609
   least_misery   -> Casa.cafe                          score=0.6239
   most_pleasure  -> Casa.cafe                          score=0.8488
   fair           -> Casa.cafe                          score=0.7571


### Reading it
The four selectors are Vincent's code, unmodified, run on real menu data — and
the numeric scores clearly differ (Casa.cafe scores 0.624 under least_misery but
0.849 under most_pleasure). What is notable is that **the argmax coincides across
all four** on this catalog: every selector picks Casa.cafe. That is not a bug in
the selectors, it is a property of a catalog where 5 restaurants score within a
narrow band of each other. The constructed counter-example in **03-3** shows the
selectors picking DIFFERENT winners once a real conflict exists in the data —
proving the machinery works even though this catalog does not yet exercise it.
